# K3 · Multi-domain figure classifier: does ViT work beyond computer-science figures?
1. Restore the SPIQA figures (computer science) from your private Hugging Face cache (~1 min).
2. Stream a few BIOMEDICA shards (biomedical figures from PubMed Central, Commercial-Use licences) and keep a
   class-balanced sample: up to 800 train / 100 val / 300 test figures per class (~10-15 min, CPU/network).
3. **Before:** test the SPIQA-only ViT on SPIQA test **and** biomedical test.
4. **Train** ViT on SPIQA + biomedical figures (same settings as before, seed 42, ~25 min).
5. **After:** test the multi-domain ViT the same way.

Needs: GPU T4, Internet on, `HF_TOKEN` secret, and the BIOMEDICA terms accepted on Hugging Face.

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. SPIQA figures from the Hugging Face cache

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(); HF_USER = api.whoami()["name"]
if not os.path.exists("/content/data/spiqa_figs/manifest.csv"):
    hf_hub_download(f"{HF_USER}/researchlens-spiqa-figs", "spiqa_figs.zip", repo_type="dataset", local_dir="/content/data")
    !cd /content/data && unzip -q -o spiqa_figs.zip && rm -f spiqa_figs.zip
!head -2 /content/data/spiqa_figs/manifest.csv

## 2. Biomedical figures from BIOMEDICA (~10-15 min)

In [ ]:
!python -m researchlens.vision.prepare_biomedica_figures --spiqa_dir /content/data/spiqa_figs --out_dir /content/data/figs_multidomain --n_train 800 --n_val 100 --n_test 300 --max_shards 6

## 3. Before: the SPIQA-only ViT on both domains

In [ ]:
!python -m researchlens.vision.eval_figcls_crossdomain --model Kavinvelavan/researchlens-vit-figure-type --data_dir /content/data/figs_multidomain --run_name vit_spiqa_only

## 4. Train the multi-domain ViT (~25 min)

In [ ]:
run("researchlens.vision.train_figcls", "configs/v2_figcls_multidomain.yaml", "vit_multidomain", 42, "--push_to_hub")

## 5. After: the multi-domain ViT on both domains

In [ ]:
!python -m researchlens.vision.eval_figcls_crossdomain --model /tmp/rl/final_models/v2_figcls/vit_multidomain/seed_42 --data_dir /content/data/figs_multidomain --run_name vit_multidomain

In [ ]:
import json
R = f"{DRIVE_ROOT}/results/v2_figcls_crossdomain"
b = json.load(open(f"{R}/vit_spiqa_only/seed_42/metrics.json"))
a = json.load(open(f"{R}/vit_multidomain/seed_42/metrics.json"))
print(f"{'test set':<26}{'metric':<10}{'SPIQA-only':>12}{'multi-domain':>14}")
for d, name in (("spiqa", "SPIQA (comp. science)"), ("biomedical", "BIOMEDICA (biomedical)")):
    for m in ("accuracy", "f1_macro"):
        print(f"{name:<26}{m:<10}{b[f'test_{d}_{m}']*100:>11.1f}%{a[f'test_{d}_{m}']*100:>13.1f}%")
print(f"{'drop CS -> biomedical':<26}{'macro-F1':<10}{b['domain_drop_f1_macro']*100:>11.1f} {a['domain_drop_f1_macro']*100:>12.1f}  (points)")

## Back up all results to Hugging Face (Kaggle has no Drive)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(); user = api.whoami()["name"]
repo = f"{user}/researchlens-results"
api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=f"{DRIVE_ROOT}/results", path_in_repo="results", repo_id=repo, repo_type="dataset")
print(f"results backed up to https://huggingface.co/datasets/{repo} (private)")